# struct-extract-eval: End-to-End Demo

This notebook walks through the full evaluation workflow:

1. **Create data** -- synthetic materials science gold + extracted pairs
2. **Infer schema** -- derive a resolved schema from gold instances
3. **Generate eval schema** -- add default `x-eval-*` comparators
4. **Edit** -- customize comparators, transforms, skip, semantic
5. **Validate gold** -- check required fields before evaluation
6. **Register semantic comparator** -- GroqJudge (real LLM) for free-text fields
7. **Evaluate** -- run the pipeline
8. **Inspect** -- run summary, per-field breakdown, per-record detail

**Requirements:**
- `pip install -e '.[dev,batch]'`
- `GROQ_API_KEY` environment variable (free at https://console.groq.com)

## Helpers: display utilities

In [ ]:
import json
import difflib
from copy import deepcopy
from html import escape
from IPython.display import HTML, display


def show_json(data: dict, title: str = "") -> None:
    """Display a JSON object with syntax highlighting."""
    text = escape(json.dumps(data, indent=2, ensure_ascii=False))
    header = f"<h4>{escape(title)}</h4>" if title else ""
    display(HTML(f"""{header}<pre style="background:#ffffff; color:#000000; padding:12px; border-radius:6px;
        border:1px solid #d0d7de; font-size:13px; line-height:1.5;
        max-height:500px; overflow:auto;">{text}</pre>"""))


def show_diff(before: dict, after: dict, before_title: str = "Before", after_title: str = "After") -> None:
    """Show a unified diff of two JSON objects, color-coded like git diff."""
    before_lines = json.dumps(before, indent=2, ensure_ascii=False).splitlines()
    after_lines = json.dumps(after, indent=2, ensure_ascii=False).splitlines()
    diff = list(difflib.unified_diff(before_lines, after_lines, lineterm="",
                                      fromfile=before_title, tofile=after_title, n=3))
    if not diff:
        display(HTML("<p><em>No differences</em></p>"))
        return
    html_lines = []
    for line in diff:
        escaped = escape(line)
        if line.startswith("+++") or line.startswith("---"):
            html_lines.append(f'<span style="color:#656d76; font-weight:bold;">{escaped}</span>')
        elif line.startswith("@@"):
            html_lines.append(f'<span style="color:#8250df;">{escaped}</span>')
        elif line.startswith("+"):
            html_lines.append(f'<span style="background:#dafbe1; color:#1a7f37;">{escaped}</span>')
        elif line.startswith("-"):
            html_lines.append(f'<span style="background:#ffebe9; color:#cf222e;">{escaped}</span>')
        else:
            html_lines.append(f'<span style="color:#000000;">{escaped}</span>')
    joined = "\n".join(html_lines)
    display(HTML(f"""<pre style="background:#ffffff; color:#000000; padding:12px; border-radius:6px;
        border:1px solid #d0d7de; font-size:13px; line-height:1.6;
        max-height:600px; overflow:auto;">{joined}</pre>"""))


def show_pair(gold: dict, extracted: dict, record_id: str | int = "") -> None:
    """Show gold vs extracted side by side."""
    gold_text = escape(json.dumps(gold, indent=2, ensure_ascii=False))
    ext_text = escape(json.dumps(extracted, indent=2, ensure_ascii=False))
    title = f"<h4>{escape(str(record_id))}</h4>" if record_id != "" else ""
    display(HTML(f"""{title}
    <div style="display:flex; gap:16px;">
        <div style="flex:1;">
            <div style="font-weight:bold; margin-bottom:4px; color:#000000;">Gold</div>
            <pre style="background:#ffffff; color:#000000; padding:12px; border-radius:6px;
                border:1px solid #d0d7de; font-size:13px; line-height:1.5;
                max-height:400px; overflow:auto;">{gold_text}</pre>
        </div>
        <div style="flex:1;">
            <div style="font-weight:bold; margin-bottom:4px; color:#000000;">Extracted</div>
            <pre style="background:#ffffff; color:#000000; padding:12px; border-radius:6px;
                border:1px solid #d0d7de; font-size:13px; line-height:1.5;
                max-height:400px; overflow:auto;">{ext_text}</pre>
        </div>
    </div>"""))


def show_table(headers: list[str], rows: list[list]) -> None:
    """Render a table with color coding for status columns."""
    status_colors = {
        "match": "#1a7f37", "mismatch": "#cf222e",
        "omission": "#9a6700", "hallucination": "#8250df",
        "skipped": "#656d76", "pending": "#0969da", "batch_error": "#cf222e",
    }
    th = "".join(
        f"<th style='text-align:left; padding:6px 12px; border-bottom:2px solid #d0d7de; background:#ffffff; color:#000000;'>{escape(str(h))}</th>"
        for h in headers
    )
    trs = []
    for row in rows:
        tds = []
        for val in row:
            val_str = escape(str(val))
            color = status_colors.get(str(val), "#000000")
            style = f"color:{color}; font-weight:bold;" if str(val) in status_colors else f"color:#000000;"
            tds.append(f"<td style='padding:4px 12px; border-bottom:1px solid #eaeef2; background:#ffffff; {style}'>{val_str}</td>")
        trs.append(f"<tr>{''.join(tds)}</tr>")
    tbody = "".join(trs)
    display(HTML(f"""<table style="border-collapse:collapse; font-size:13px; font-family:monospace;">
        <thead><tr>{th}</tr></thead><tbody>{tbody}</tbody></table>"""))

## 1. Synthetic Data

3 records of materials science extraction covering key scenarios:

1. **Synonym + hallucination** -- method synonym (CVD), extra hallucinated step
2. **Omission + wrong value** -- missing step, wrong temperature, missing optional field
3. **Perfect match** -- everything correct, optional fields absent in both

In [ ]:
GOLD = [
    {
        "method": "Chemical Vapor Deposition",
        "temperature": 773.15,
        "lab_id": "LAB-001",
        "description": "Deposited thin film at high temperature.",
        "observation": "Stable film with uniform coverage.",
        "outcome": "Success, target efficiency achieved.",
        "steps": [
            {"name": "deposit", "duration": 120},
            {"name": "anneal", "duration": 60},
        ],
    },
    {
        "method": "Sputtering",
        "temperature": 500.0,
        "lab_id": "LAB-002",
        "description": "Sputtered target onto substrate.",
        "observation": "Thin but patchy coating observed.",
        "outcome": "Partial success, some defects present.",
        "steps": [
            {"name": "clean", "duration": 30},
            {"name": "deposit", "duration": 90},
        ],
    },
    {
        # No lab_id, no description -> makes them optional in inferred schema
        "method": "Pulsed Laser Deposition",
        "temperature": 700.0,
        "observation": "Smooth crystalline film.",
        "outcome": "Fully successful.",
        "steps": [
            {"name": "ablate", "duration": 45},
            {"name": "deposit", "duration": 90},
        ],
    },
]

EXTRACTED = [
    {
        # method synonym, extra hallucinated step
        "method": "CVD",
        "temperature": 773.15,
        "lab_id": "LAB-001",
        "description": "Film deposited using CVD process.",
        "observation": "Film is stable with even coating.",          # paraphrase of gold
        "outcome": "Experiment succeeded, reached target efficiency.", # paraphrase of gold
        "steps": [
            {"name": "deposit", "duration": 120},
            {"name": "anneal", "duration": 60},
            {"name": "cool", "duration": 30},  # hallucinated
        ],
    },
    {
        # temperature wrong (450 vs 500), missing step, missing lab_id
        "method": "Sputtering",
        "temperature": 450.0,
        "description": "Standard sputtering procedure.",
        "observation": "Coating was thin and had visible patches.",  # paraphrase
        "outcome": "Film had major defects throughout.",              # NOT a paraphrase -- factual disagreement
        "steps": [
            {"name": "clean", "duration": 30},
            # missing "deposit" step -> omission
        ],
    },
    {
        # perfect match
        "method": "Pulsed Laser Deposition",
        "temperature": 700.0,
        "observation": "Smooth crystalline film.",   # exact match -> short-circuits
        "outcome": "Fully successful.",               # exact match -> short-circuits
        "steps": [
            {"name": "ablate", "duration": 45},
            {"name": "deposit", "duration": 90},
        ],
    },
]

### Gold vs Extracted: side-by-side

In [ ]:
for i in range(len(GOLD)):
    show_pair(GOLD[i], EXTRACTED[i], record_id=i)

## 2. Infer Schema from Gold Instances

`infer_schema()` examines all gold records and produces a resolved JSON Schema.
Fields absent in some instances are excluded from the `required` array.

In [ ]:
from struct_extract_eval import infer_schema, generate_eval_schema, evaluate, validate_gold

resolved_schema = infer_schema(GOLD)
show_json(resolved_schema, "Resolved Schema (inferred from gold)")
print("Note: lab_id and description are NOT in 'required' (absent in record 2).")

## 3. Generate Eval Schema

`generate_eval_schema()` infers the schema and adds `x-eval-compare` to each leaf field
based on its type, and converts the `required` array into per-field `x-eval-required: false`.

In [ ]:
eval_schema = generate_eval_schema(schema=resolved_schema)
show_diff(resolved_schema, eval_schema,
          before_title="Resolved Schema",
          after_title="Eval Schema (with x-eval-* defaults)")

## 4. User Edits the Eval Schema

The defaults are a best guess. In practice you'd save the eval schema to a file,
edit it, and load it back. Here we simulate edits in code:

- **method**: `exact` -> `oneof` with known synonyms (CVD, PLD, etc.)
- **temperature**: `numeric` (no tolerance) -> `numeric` with 1% relative tolerance
- **description**: `exact` -> `x-eval-skip: true` (free text, can't judge without domain knowledge)
- **steps[].name**: add `lowercase` + `strip` transforms before comparison

The `observation` and `outcome` fields are left at the default (`exact`) for now.
We'll switch them to `semantic` in the next step to show batch LLM judging.

In [ ]:
before_edit = deepcopy(eval_schema)
props = eval_schema["properties"]

# method: oneof with known synonyms
props["method"]["x-eval-compare"] = {
    "oneof": {"values": [
        "Chemical Vapor Deposition", "CVD",
        "Sputtering", "Sputter Deposition",
        "Pulsed Laser Deposition", "PLD",
    ]}
}

# temperature: 1% relative tolerance
props["temperature"]["x-eval-compare"] = {
    "numeric": {"tolerance": {"rel": 0.01}}
}

# description: skip (excluded from scoring)
props["description"]["x-eval-skip"] = True

# steps[].name: lowercase + strip transforms
step_items = props["steps"]["items"]
step_items["properties"]["name"]["x-eval-transform"] = ["lowercase", "strip"]

show_diff(before_edit, eval_schema,
          before_title="Eval Schema (defaults)",
          after_title="Eval Schema (after user edits)")

## 5. Validate Gold

`validate_gold()` checks that all required fields are present in every gold record.
Call this before evaluation to catch data quality issues early.

In [ ]:
validate_gold(GOLD, eval_schema)
print("Gold validation passed.")

## 6. Apply Semantic Comparator to `observation` and `outcome`

The `semantic` comparator is a **batch comparator**. It is not registered by default -- you
choose which judge to use and register it explicitly.

`GroqJudge` calls the Groq API with Llama 3.3 70B. It requires the `groq` package
and a `GROQ_API_KEY` environment variable.

### Why we apply it to TWO fields

Both `observation` and `outcome` are free-text fields where synonyms and paraphrases
should still count as correct. Using the same `semantic` comparator on both lets us
showcase the **batch grouping** behavior:

- For each record, ALL fields tagged `x-eval-compare: "semantic"` are collected first
- Then the semantic handler is called ONCE with both fields together
- That one call bundles the LLM prompt for both `observation` and `outcome`
- Result: 1 LLM call per record instead of 2 (halves the API cost and latency)

If a pair is an exact match (record 2's `observation` and `outcome`), the semantic
comparator short-circuits and doesn't bother the LLM at all.

We keep `description` as `x-eval-skip: true` -- some fields just aren't worth evaluating.

In [ ]:
# Switch observation and outcome to semantic. Leave description as skipped.
before_semantic = deepcopy(eval_schema)
props["observation"]["x-eval-compare"] = "semantic"
props["outcome"]["x-eval-compare"] = "semantic"

show_diff(before_semantic, eval_schema,
          before_title="Before (observation/outcome: exact)",
          after_title="After (observation/outcome: semantic, batched)")

In [ ]:
from struct_extract_eval.batch import GroqJudge, SemanticBatchComparator
from struct_extract_eval.core.comparators.registry import _registry, register

# Clear any previous registration (for notebook re-runs)
_registry.pop("semantic", None)

judge = GroqJudge(api_key="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")
register("semantic", SemanticBatchComparator(judge))
print("Registered 'semantic' comparator with GroqJudge (Llama 3.3 70B).")


## 7. Evaluate

In [ ]:
run = evaluate(GOLD, EXTRACTED, schema=eval_schema)

## 8. Inspect Results

### Run Summary

In [ ]:
show_table(
    ["Metric", "Value"],
    [
        ["Records", run.total_records],
        ["Fields scored", run.total_fields],
        ["Precision", f"{run.mean_precision:.3f}"],
        ["Recall", f"{run.mean_recall:.3f}"],
        ["F1", f"{run.mean_f1:.3f}"],
        ["Omissions", run.total_omissions],
        ["Hallucinations", run.total_hallucinations],
    ],
)

### Per-Field Breakdown

Which fields does the extractor struggle with?

In [ ]:
show_table(
    ["Field Path", "Mean Score", "Matches", "Mismatches", "Omissions", "Hallucinations"],
    [
        [path, f"{agg.mean_score:.2f}", agg.matches, agg.mismatches, agg.omissions, agg.hallucinations]
        for path, agg in sorted(run.per_field.items())
    ],
)

### Per-Record Detail

Drill into each record to see field-by-field results.

In [ ]:
for record in run.records:
    display(HTML(f"<h4>Record {record.record_id} &mdash; "
                 f"F1: {record.f1:.3f} &nbsp; P: {record.precision:.3f} &nbsp; R: {record.recall:.3f}</h4>"))

    show_pair(record.gold, record.extracted)

    show_table(
        ["Field", "Gold", "Extracted", "Score", "Status", "Reason"],
        [
            [fr.path, str(fr.gold_value)[:30], str(fr.extracted_value)[:30],
             f"{fr.score:.1f}", fr.status, fr.reason or ""]
            for fr in record.field_results
        ],
    )